# MAB-PDF: server AI uji (Colab GPU T4)
Notebook ini menjalankan model bahasa di GPU Colab dan membuka alamat HTTPS sementara agar MAB-PDF bisa memakainya.

**Sebelum mulai:** menu *Runtime → Change runtime type → T4 GPU*. Lalu jalankan sel dari atas ke bawah.

**Catatan penting**
- Hanya untuk uji pribadi dan singkat. Sesi Colab berhenti sendiri dan Colab gratis punya batasan pemakaian; baca ketentuannya. Bukan server produksi.
- Alamat `trycloudflare.com` tidak memakai kata sandi. Anggap alamat itu rahasia, jangan dibagikan, dan hentikan sesi bila selesai.
- Dokumen yang dirangkum lewat server ini **teksnya dikirim** ke Colab (bukan file PDF-nya).

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print("Ollama terpasang")

In [ ]:
# Pilih model. 7b = seimbang untuk T4. 14b lebih pintar tapi sekitar 2x lebih lambat. 3b lebih cepat tapi kurang rapi.
MODEL = "qwen2.5:7b"

import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_ORIGINS="*", OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="-1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server Ollama tidak mau menyala:\n" + open("/content/ollama.log").read()[-800:])
print("Server Ollama aktif")
!ollama pull $MODEL

In [ ]:
import json
def ask(prompt):
    req = urllib.request.Request("http://127.0.0.1:11434/v1/chat/completions",
        data=json.dumps({"model": MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": 80}).encode(),
        headers={"Content-Type": "application/json"})
    t = time.time()
    r = json.load(urllib.request.urlopen(req, timeout=300))
    return r["choices"][0]["message"]["content"], round(time.time() - t, 1)

print(ask("Jawab singkat: apa ibu kota Indonesia?"))   # panggilan pertama lebih lambat (memuat model ke GPU)
print(ask("Sebutkan tiga manfaat membaca, satu kalimat masing-masing."))

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import re
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "127.0.0.1:11434", "--no-autoupdate"],
                       stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
if not url:
    raise RuntimeError("Tunnel gagal:\n" + open("/content/tunnel.log").read()[-800:])
print("=" * 60)
print("Tempel di MAB-PDF -> Alamat server AI:\n  ", url)
print("Nama model:", MODEL)
print("=" * 60)

In [ ]:
# Pantau GPU saat MAB-PDF merangkum (jalankan ulang sel ini kapan saja)
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv